# CrowdHumanでYOLO人物検出モデルをファインチューニング

CrowdHumanの **vbox（人物の見えている範囲）** を使い、`person` の1クラスの検出モデルを作成します。

このノートブックでは、まずGPU・Google Drive・保存済みファイルだけを確認します。
各セルを上から順に実行してください。**セル4の出力を確認してから、次の処理を追加します。**
画像展開、ZIP内容の検証、注釈解析、ラベル変換、モデル取得、学習はまだ行いません。

- データ保存場所: `/content/drive/MyDrive/crowdhuman`
- 学習結果の保存先: `/content/drive/MyDrive/crowdhuman_vbox_ft/runs`
- 事前学習重みの保存先: `/content/drive/MyDrive/crowdhuman_vbox_ft/weights`
- 今後のローカルデータ展開先: `/content/crowdhuman_vbox_yolo`（現段階では作成しません）


## 1. GPUを確認

Colabの「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してから実行してください。
`nvidia-smi` の結果と、PyTorchから見えるGPU・メモリ・バージョンを表示します。


In [ ]:
import subprocess
import torch

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
except FileNotFoundError:
    print("nvidia-smiが見つかりません。ColabのGPU設定を確認してください。")

print(f"PyTorch: {torch.__version__}")
print(f"PyTorchのCUDAバージョン: {torch.version.cuda}")
print(f"CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {properties.name}")
        print(f"GPU {index}の総メモリ: {properties.total_memory / 1024**3:.2f} GiB")
else:
    print("GPUが利用できません。ランタイムのタイプを確認してください。")


## 2. Google Driveを接続

画面の案内に従って、CrowdHumanを保存しているGoogleアカウントのDriveを接続してください。


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. 保存先とColabの空き容量を確認

学習結果用と事前学習重み用のフォルダを作成します。既存のファイルは削除しません。
データ保存フォルダとローカルデータ展開先は、このセルでは作成しません。
表示する空き容量は **Colabローカル** の容量です。


In [ ]:
from pathlib import Path
import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR = DRIVE_ROOT / "crowdhuman"
OUTPUT_ROOT = DRIVE_ROOT / "crowdhuman_vbox_ft"
RUNS_DIR = OUTPUT_ROOT / "runs"
WEIGHTS_DIR = OUTPUT_ROOT / "weights"
LOCAL_DATA_DIR = Path("/content/crowdhuman_vbox_yolo")

if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Driveが接続されていません。先にセル2を実行してください。")

for directory in (RUNS_DIR, WEIGHTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"データ保存場所: {DATA_DIR}")
print(f"学習結果の保存先: {RUNS_DIR}")
print(f"事前学習重みの保存先: {WEIGHTS_DIR}")
print(f"今後のローカルデータ展開先: {LOCAL_DATA_DIR}")
usage = shutil.disk_usage("/content")
print(f"Colabローカルの総容量: {usage.total / 1024**3:.2f} GiB")
print(f"Colabローカルの空き容量: {usage.free / 1024**3:.2f} GiB")


## 4. 展開済みCrowdHumanのフォルダ構成を確認

データフォルダ直下に `checkpoints`、`logs`、`prepared`、`raw` があることを確認済みです。
画像と注釈の保存場所を調べるため、`raw` と `prepared` の中を最大20件ずつ表示します。
さらに、表示された子フォルダのうち最大5個について、中身を最大5件ずつ表示します。
表示順はファイルシステムの順序で、上限に達した場合は全件を表示していません。

このセルは一覧とファイルサイズの確認のみです。画像・注釈の内容の読み込み、データの変更は行いません。
`checkpoints` と `logs` の中身は確認しません。
**実行結果を共有し、保存場所を確認できるまでここで止めてください。**


In [ ]:
from itertools import islice


def show_entries(directory, limit, indent="  "):
    # 大量の画像があっても、一覧取得は上限+1件で止めます。
    entries = list(islice(directory.iterdir(), limit + 1))
    shown = entries[:limit]
    for entry in shown:
        if entry.is_dir():
            print(f"{indent}[フォルダ] {entry.name}")
        elif entry.is_file():
            print(f"{indent}[ファイル] {entry.name} ({entry.stat().st_size:,} bytes)")
        else:
            print(f"{indent}[その他] {entry.name}")
    if not entries:
        print(f"{indent}（空のフォルダです）")
    if len(entries) > limit:
        print(f"{indent}（表示上限{limit}件に達しました。続きは省略しています）")
    return shown


if not DATA_DIR.is_dir():
    print(f"データフォルダが見つかりません: {DATA_DIR}")
    print("Driveの接続状態と保存場所を確認してください。")
else:
    for name in ("raw", "prepared"):
        directory = DATA_DIR / name
        print(f"\n確認先: {directory}")
        if not directory.is_dir():
            print("  フォルダが見つかりません。")
            continue
        shown = show_entries(directory, limit=20)
        children = [entry for entry in shown if entry.is_dir()]
        for child in children[:5]:
            print(f"\n  子フォルダ: {child}")
            show_entries(child, limit=5, indent="    ")
        if len(children) > 5:
            print("  （子フォルダの詳細表示は最大5個までです）")

print("\nここで実行結果を共有してください。画像と注釈の保存場所を確認します。")
